In [1]:
import logging

import httpx
import pydantic

from seis_lab_data.authentik import get_user_by_uuid
from seis_lab_data.config import get_settings
from seis_lab_data.db import queries
from seis_lab_data.operations import discovery as discovery_ops
from seis_lab_data.schemas import (
    discovery as discovery_schemas,
    projects as project_schemas,
)

logging.basicConfig(level=logging.DEBUG)
logging.getLogger("httpcore").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

web_client = httpx.AsyncClient()
settings = get_settings()

In [2]:
async with settings.get_db_session_maker()() as session:
    p1 = await queries.get_project_by_english_name(session, "PRR windfarms")
    assert p1 is not None

In [3]:
discovery_owner = await get_user_by_uuid(
    admin_token=settings.auth_admin_token,
    user_id=p1.owner_id,
    web_client=web_client,
    authentik_base_url=settings.auth_internal_base_url,
)
assert discovery_owner is not None

In [4]:
async with settings.get_db_session_maker()() as session:
    new_missions = await discovery_ops.discover_project_contents(
        session=session,
        project=p1,
        settings=settings,
        user=discovery_owner,
    )

DEBUG:seis_lab_data.operations.discovery:mission_path=Path('/data/simulated-archive/surveys/owf-seism-2024')
DEBUG:seis_lab_data.events.emitters:no-op emit event called with event=SeisLabDataEvent(type_=<EventType.SURVEY_MISSION_CREATED: 'survey_mission_created'>, initiator='94291116-7fb7-4889-9d83-8b04718d0248', payload=EventPayload(before=None, after={'id': '36e79217-3578-45d7-bd87-f4642d95ae46', 'name': {'en': 'seism-2024', 'pt': None}, 'description': {'en': 'Some description about the seism-2024 survey', 'pt': None}, 'status': <SurveyMissionStatus.DRAFT: 'draft'>, 'validation_result': None, 'project': {'id': '2b663029-2651-4c2f-b4a9-2f53d5d00c41', 'name': {'en': 'PRR windfarms', 'pt': 'PRR Eólicas'}, 'status': <ProjectStatus.DRAFT: 'draft'>, 'validation_result': None, 'root_path': 'surveys', 'bbox_4326': None, 'temporal_extent_begin': '', 'temporal_extent_end': '', 'num_survey_missions': 0, 'num_survey_related_records': 0}, 'temporal_extent_begin': '', 'temporal_extent_end': '', 'b

SeisLabDataError: There is already a survey-related record with english name 'Raw bathymetry' for the same survey mission.

In [5]:
new_missions

[]

In [18]:
new_missions[0].project.discovery_configuration

{'links': None,
 'records': {'raw_bathy': {'id_': 'raw_bathy',
   'name': {'en': 'Raw bathymetry', 'pt': 'Batimetria em bruto'},
   'links': None,
   'assets': [{'name': {'en': 'kmall file', 'pt': 'Ficheiro kmall'},
     'links': None,
     'description': None,
     'extra_properties': None,
     'discovery_patterns': ['s06-mbes/s02-raw-data/01-EM712/{{date_dashed}}/\\d{4}_{{date}}_\\d{6}_{{ship}}.kmall']}],
   'description': None,
   'domain_type': 'geophysical',
   'workflow_stage': 'raw data',
   'dataset_category': 'bathymetry',
   'extra_properties': [{'handler': {'type': 'date:ymd',
      'pattern': '\\d{8}',
      'compatibility': 'day'},
     'identifier': 'date'},
    {'handler': {'type': 'date:ymd-dashed',
      'pattern': '\\d{4}-\\d{2}-\\d{2}',
      'compatibility': 'day'},
     'identifier': 'date_dashed'},
    {'handler': {'type': 'constant',
      'choices': None,
      'pattern': '\\w+',
      'compatibility': 'equal'},
     'identifier': 'ship'}],
   'metadata_extract

In [6]:
p1.discovery_configuration["records"]["raw_bathy"]["extra_properties"][0]["handler"][
    "type_"
] = "date:ymd"

In [7]:
pydantic.TypeAdapter(discovery_schemas.PropertyHandler).validate_python(
    p1.discovery_configuration["records"]["raw_bathy"]["extra_properties"][0]["handler"]
)

DateYmdProperty(type_='date:ymd', pattern='\\d{8}', compatibility='day')

In [ ]:
project_schemas.ProjectReadDetail.from_db_instance(p1).discovery_configuration